In [ ]:
# print("123")

# HOMEWORK WEEK-4

In [1]:
!pip install gitsource

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [gitsource]

[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [4]:
import os
import json
from evaluation_utils import llm_structured
from gitsource import GithubRepositoryDataReader
from pydantic import BaseModel
from dotenv import load_dotenv
load_dotenv()

True

In [ ]:
from google import genai
from google.genai import types
client = genai.Client(api_key=os.getenv("GEMINI_API_KEY"))

In [ ]:
model='gemini-3.5-flash'

In [ ]:
reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [ ]:
documents = [file.parse() for file in reader.read()]

In [ ]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [ ]:
user_prompt = """
FILENAME: {filename}

LESSON CONTENT:
{content}
""".strip()

In [ ]:
class Question(BaseModel):
    questions: list[str]

In [ ]:
messages = [
    types.Content(
        role="user",
        parts=[
            types.Part.from_text(text=user_prompt), 
            types.Part.from_text(text=data_gen_instructions)
        ]
    )
]

config = types.GenerateContentConfig(
    temperature=0.2,
    max_output_tokens=2000,
    response_mime_type="application/json",  # Forces Gemini to output strict JSON
    response_schema=Question                # Enforces your structural blueprint
)

response = client.models.generate_content(
    model=model,
    contents=messages,
    config=config
)

## QUESTION 1

In [ ]:
input_tokens = []

In [ ]:
# usage = response.usage_metadata

In [ ]:
for doc in documents[:3]:
    user_prompt_template = user_prompt.format(
        filename=doc["filename"],
        content=doc["content"],
    )

    parsed, usage = llm_structured(client, data_gen_instructions, user_prompt_template, Question)

    print(doc["filename"], "->", usage.prompt_token_count, "input tokens")
    input_tokens.append(response.usage_metadata)

average_input_tokens = sum(input_tokens) / len(input_tokens)
print("\nAverage input tokens:", average_input_tokens)

## QUESTION 2